# TP 6 — XGBoost (gradient boosting d’arbres)
**Machine Learning · Apprentissage supervisé · Ensemble d’arbres (boosting)**

Ce TP reprend les exemples de la présentation (diapos 46 à 52) :

| Exemple | Diapos | Problème |
|---|---|---|
| Exemple 1 | 47 – 48 | Régression : ventes de glaces selon la température, 3 arbres |
| Exemple 2 | 49 | Classification : réussite d’un étudiant selon ses heures de révision, 3 arbres |
| Exemple 3 | 50 | Effet du paramètre λ (régularisation des feuilles) |

**Objectifs** : programmer le boosting **en Python pur** (prédiction de départ, résidus, petit arbre, valeur des feuilles, mise à jour avec η) ; l’appliquer aux exemples des diapos ; puis refaire les mêmes exemples avec la bibliothèque **xgboost** et comparer les deux implémentations (`XGBRegressor`, `XGBClassifier`).

## Rappel de cours

1. **Départ** : $F_0$ = moyenne des $y$ (régression) ; $F_0 = 0$, donc $p = 0{,}5$ (classification, en *log-odds*).
2. **Résidus** (ce qui reste à corriger) : $r = y - F$ (régression) ; $r = y - p$ avec $p = \sigma(F)$ (classification).
3. **Petit arbre** (ici une seule question, `max_depth = 1`) ajusté sur les résidus. Valeur d’une feuille :
$$w = \frac{\sum r}{n + \lambda}\ \ (\text{régression}) \qquad w = \frac{\sum (y - p)}{\sum p(1-p) + \lambda}\ \ (\text{classification})$$
4. **Mise à jour** : $F := F + \eta \times w$, puis on recommence avec l’arbre suivant.

**Comment XGBoost choisit la question d’un arbre ?** On note $G = \sum r$ et $H$ = nombre d’exemples (régression) ou $\sum p(1-p)$ (classification). La qualité d’une question est
$$\text{Gain} = \frac{G_{\text{gauche}}^2}{H_{\text{gauche}} + \lambda} + \frac{G_{\text{droite}}^2}{H_{\text{droite}} + \lambda} - \frac{G^2}{H + \lambda}$$
et l’arbre garde la question au **plus grand gain** (c’est l’équivalent de la plus petite SSE vue au TP 4).

In [ ]:
import math

---
# Partie A — Implémentation from scratch (Python pur)

## Exemple 1 : ventes de glaces

| T (°C) | 10 | 20 | 30 | 40 |
|---|---|---|---|---|
| Ventes $y$ (TND) | 10 | 20 | 60 | 70 |

Réglages : 3 arbres, une question par arbre, $\eta = 0{,}5$, $\lambda = 0$.

In [ ]:
T = [10, 20, 30, 40]
Y = [10, 20, 60, 70]

def seuils(X):
    v = sorted(set(X))
    return [(a + b) / 2 for a, b in zip(v, v[1:])]

def meilleure_question(X, G, H, lam):
    # G[i] = résidu de l'exemple i ; H[i] = 1 (régression) ou p(1-p) (classification)
    # Renvoie (gain, seuil) pour chaque seuil testé
    G_tot, H_tot = sum(G), sum(H)
    resultats = []
    for s in seuils(X):
        Gg = sum(g for x, g in zip(X, G) if x <= s); Hg = sum(h for x, h in zip(X, H) if x <= s)
        Gd, Hd = G_tot - Gg, H_tot - Hg
        gain = Gg ** 2 / (Hg + lam) + Gd ** 2 / (Hd + lam) - G_tot ** 2 / (H_tot + lam)
        resultats.append((gain, s))
    return resultats

def feuilles(X, G, H, s, lam):
    Gg = sum(g for x, g in zip(X, G) if x <= s); Hg = sum(h for x, h in zip(X, H) if x <= s)
    Gd = sum(g for x, g in zip(X, G) if x > s); Hd = sum(h for x, h in zip(X, H) if x > s)
    return Gg / (Hg + lam), Gd / (Hd + lam)

### Question 1 — Arbre 1 : départ, Calcul de  résidus, Gain de question et feuilles

- $F_0 = (10 + 20 + 60 + 70)/4 = 40$ → résidus $r_0 = -30 ; -20 ; 20 ; 30$, MSE = 650
- Question « T ≤ 25 ? » : feuille gauche $(-30 - 20)/2 = -25$, feuille droite $(20 + 30)/2 = 25$
- $F_1 = 40 + 0{,}5 \times (-25) = 27{,}5$ à gauche et $40 + 0{,}5 \times 25 = 52{,}5$ à droite

In [ ]:
eta, lam = 0.5, 0.0
F = [sum(Y) / len(Y)] * len(Y)
mse = lambda F: sum((y - f) ** 2 for y, f in zip(Y, F)) / len(Y)

r0 = [y - f for y, f in zip(Y, F)]
print("F0 =", F[0], "| MSE de départ =", mse(F))
print("Résidus r0 :", r0)
gains = meilleure_question(T, r0, [1] * 4, lam)
print("Gains des questions :", [(f"T ≤ {s}", round(g, 2)) for g, s in gains])
g1, s1 = max(gains)
wg, wd = feuilles(T, r0, [1] * 4, s1, lam)
print(f"Question retenue : T ≤ {s1} ; feuilles {wg} / {wd}")
F = [f + eta * (wg if t <= s1 else wd) for t, f in zip(T, F)]
print("F1 =", F, "| MSE =", mse(F))

F0 = 40.0 | MSE de départ = 650.0
Résidus r0 : [-30.0, -20.0, 20.0, 30.0]
Gains des questions : [('T ≤ 15.0', 1200.0), ('T ≤ 25.0', 2500.0), ('T ≤ 35.0', 1200.0)]
Question retenue : T ≤ 25.0 ; feuilles -25.0 / 25.0
F1 = [27.5, 27.5, 52.5, 52.5] | MSE = 181.25


### Question 2 — Arbre 2 : La correction ce qui reste
Résidus $r_1 = -17{,}5 ; -7{,}5 ; 7{,}5 ; 17{,}5$. La meilleure question est encore « T ≤ 25 ? » (gain 625). Feuilles $-12{,}5$ et $12{,}5$ → $F_2 = 21{,}25$ et $58{,}75$ ; MSE = 64,1.

In [ ]:
r1 = [y - f for y, f in zip(Y, F)]
gains = meilleure_question(T, r1, [1] * 4, lam)
print("Résidus r1 :", r1, "| gains :", [(f"T ≤ {s}", round(g, 2)) for g, s in gains])
g2, s2 = max(gains)
wg, wd = feuilles(T, r1, [1] * 4, s2, lam)
F = [f + eta * (wg if t <= s2 else wd) for t, f in zip(T, F)]
F2 = F[:]
print(f"Question retenue : T ≤ {s2} ; feuilles {wg} / {wd} ; F2 = {F}")
print("MSE après 2 arbres =", mse(F))

Résidus r1 : [-17.5, -7.5, 7.5, 17.5] | gains : [('T ≤ 15.0', 408.33), ('T ≤ 25.0', 625.0), ('T ≤ 35.0', 408.33)]
Question retenue : T ≤ 25.0 ; feuilles -12.5 / 12.5 ; F2 = [21.25, 21.25, 58.75, 58.75]
MSE après 2 arbres = 64.0625


### Question 3 — Arbre 3 : Observer que la meilleure question change
Résidus $r_2 = -11{,}25 ; -1{,}25 ; 1{,}25 ; 11{,}25$. On calcule le gain des trois questions possibles :

| Question | $G_g$ | $G_d$ | Gain |
|---|---|---|---|
| T ≤ 15 | −11,25 (1 jour) | 11,25 (3 jours) | $126{,}56 + 42{,}19 - 0 =$ **168,75** |
| T ≤ 25 | −12,5 (2 jours) | 12,5 (2 jours) | $78{,}13 + 78{,}13 - 0 = 156{,}25$ |
| T ≤ 35 | −11,25 (3 jours) | 11,25 (1 jour) | $42{,}19 + 126{,}56 - 0 =$ **168,75** |

> ⚠️ **Attention.** « T ≤ 15 » et « T ≤ 35 » sont **à égalité** (168,75) et **meilleures** que « T ≤ 25 » (156,25) : au 3ᵉ arbre, la question **change**. La bibliothèque xgboost retient « T ≤ 35 » ; notre code fait de même (en cas d’égalité, il garde le dernier seuil). On obtient $F_3 = 19{,}38 ; 19{,}38 ; 56{,}88 ; 64{,}38$ et une MSE de 32,4.

In [ ]:
r2 = [y - f for y, f in zip(Y, F2)]
gains3 = meilleure_question(T, r2, [1] * 4, lam)
for g, s in gains3:
    print(f"T ≤ {s} : gain = {g:.2f}")

# XGBoost : plus grand gain ; en cas d'égalité on garde le dernier seuil (comme la bibliothèque ici)
meilleur = max(g for g, s in gains3)
s3 = [s for g, s in gains3 if abs(g - meilleur) < 1e-9][-1]
wg, wd = feuilles(T, r2, [1] * 4, s3, lam)
F3 = [f + eta * (wg if t <= s3 else wd) for t, f in zip(T, F2)]
print(f"\nQuestion retenue : T ≤ {s3} : feuilles {wg} / {wd} ; F3 = {F3} ; MSE = {mse(F3):.2f}")

T ≤ 15.0 : gain = 168.75
T ≤ 25.0 : gain = 156.25
T ≤ 35.0 : gain = 168.75

Question retenue : T ≤ 35.0 : feuilles -3.75 / 11.25 ; F3 = [19.375, 19.375, 56.875, 64.375] ; MSE = 32.42


### Question 4 — trouver une fonction complète de boosting, et la prédiction pour un jour chaud
 On rassemble les étapes dans une fonction qui enchaîne `n_arbres` souches. Pour un jour chaud (38 °C, au-dessus de tous les seuils), la prédiction est la valeur de droite : $F_3 \approx 64$ TND.

In [ ]:
def boosting_regression(X, Y, n_arbres, eta, lam):
    F0 = sum(Y) / len(Y)
    F = [F0] * len(Y)
    arbres, mses = [], [sum((y - f) ** 2 for y, f in zip(Y, F)) / len(Y)]
    for _ in range(n_arbres):
        r = [y - f for y, f in zip(Y, F)]
        gains = meilleure_question(X, r, [1] * len(X), lam)            # la question est recalculée à chaque arbre
        m = max(g for g, s in gains)
        s = [s for g, s in gains if abs(g - m) < 1e-9][-1]
        wg, wd = feuilles(X, r, [1] * len(X), s, lam)
        arbres.append((s, wg, wd))
        F = [f + eta * (wg if x <= s else wd) for x, f in zip(X, F)]
        mses.append(sum((y - f) ** 2 for y, f in zip(Y, F)) / len(Y))
    return F0, arbres, mses

def predire_boost(x, F0, arbres, eta):
    return F0 + sum(eta * (wg if x <= s else wd) for s, wg, wd in arbres)

F0, arbres_vrai, mses_vrai = boosting_regression(T, Y, 3, 0.5, 0.0)
for k, (s, wg, wd) in enumerate(arbres_vrai, start=1):
    print(f"Arbre {k} : T ≤ {s} ; feuilles {wg:g} / {wd:g}")
print("MSE après 0, 1, 2, 3 arbres :", [round(m, 1) for m in mses_vrai])
print(f"Prédiction pour 38 °C : {predire_boost(38, F0, arbres_vrai, 0.5):.2f} TND")

Arbre 1 : T ≤ 25.0 ; feuilles -25 / 25
Arbre 2 : T ≤ 25.0 ; feuilles -12.5 / 12.5
Arbre 3 : T ≤ 35.0 ; feuilles -3.75 / 11.25
MSE après 0, 1, 2, 3 arbres : [650.0, 181.2, 64.1, 32.4]
Prédiction pour 38 °C : 64.38 TND


## Exemple 2 : classification, réussite d’un étudiant

| Heures $x$ | 1 | 2 | 3 | 4 |
|---|---|---|---|---|
| Réussite $y$ | 0 | 0 | 1 | 1 |

Départ : $F_0 = 0$ (log-odds), donc $p = 0{,}5$ ; 3 arbres ; $\eta = 0{,}5$ ; $\lambda = 0$.

### Question 5 — Les 3 arbres
**Réponse.**
- Arbre 1 : erreurs $y - p = -0{,}5 ; -0{,}5 ; 0{,}5 ; 0{,}5$ ; $p(1-p) = 0{,}25$. Question « x ≤ 2,5 ? » : feuille gauche $(-0{,}5 - 0{,}5)/(0{,}25 + 0{,}25) = -2$, droite $+2$. $F_1 = \mp 1$ → $p = 0{,}27 ; 0{,}73$.
- Arbre 2 : feuilles ∓1,37 → $F_2 = \mp 1{,}68$ → $p = 0{,}16 ; 0{,}84$.
- Arbre 3 : feuilles ∓1,19 → $F_3 = \mp 2{,}28$ → $p = 0{,}09 ; 0{,}91$.

La valeur d’une feuille tient compte de la **courbure** $p(1-p)$ de la log-loss.

In [ ]:
Xc = [1, 2, 3, 4]
Yc = [0, 0, 1, 1]
sig = lambda z: 1 / (1 + math.exp(-z))
logloss = lambda P: -sum(y * math.log(p) + (1 - y) * math.log(1 - p) for y, p in zip(Yc, P)) / len(Yc)

def boosting_classification(X, Y, n_arbres, eta, lam):
    F = [0.0] * len(Y)
    historique = [(F[:], [sig(f) for f in F])]
    arbres = []
    for _ in range(n_arbres):
        P = [sig(f) for f in F]
        G = [y - p for y, p in zip(Y, P)]
        H = [p * (1 - p) for p in P]
        gains = meilleure_question(X, G, H, lam)
        m = max(g for g, s in gains)
        s = [s for g, s in gains if abs(g - m) < 1e-9][-1]
        wg, wd = feuilles(X, G, H, s, lam)
        arbres.append((s, wg, wd))
        F = [f + eta * (wg if x <= s else wd) for x, f in zip(X, F)]
        historique.append((F[:], [sig(f) for f in F]))
    return arbres, historique

arbres_c, hist = boosting_classification(Xc, Yc, 3, 0.5, 0.0)
for k, (s, wg, wd) in enumerate(arbres_c, start=1):
    F, P = hist[k]
    print(f"Arbre {k} : x ≤ {s} ; feuilles {wg:.3f} / {wd:.3f} ; F = {[round(f, 3) for f in F]} ; p = {[round(p, 3) for p in P]} ; perte = {logloss(P):.3f}")

print(f"Perte (log-loss) après 0, 1, 2, 3 arbres : {[round(logloss(hist[k][1]), 3) for k in range(4)]}")
p35 = sig(sum(0.5 * (wg if 3.5 <= s else wd) for s, wg, wd in arbres_c))
print(f"Étudiant de 3,5 h : p(réussite) = {p35:.2f} →", "RÉUSSITE" if p35 >= 0.5 else "ÉCHEC")

Arbre 1 : x ≤ 2.5 ; feuilles -2.000 / 2.000 ; F = [-1.0, -1.0, 1.0, 1.0] ; p = [0.269, 0.269, 0.731, 0.731] ; perte = 0.313
Arbre 2 : x ≤ 2.5 ; feuilles -1.368 / 1.368 ; F = [-1.684, -1.684, 1.684, 1.684] ; p = [0.157, 0.157, 0.843, 0.843] ; perte = 0.170
Arbre 3 : x ≤ 2.5 ; feuilles -1.186 / 1.186 ; F = [-2.277, -2.277, 2.277, 2.277] ; p = [0.093, 0.093, 0.907, 0.907] ; perte = 0.098
Perte (log-loss) après 0, 1, 2, 3 arbres : [0.693, 0.313, 0.17, 0.098]
Étudiant de 3,5 h : p(réussite) = 0.91 → RÉUSSITE


## Exemple 3 : le paramètre λ

### Question 6 — D’où vient la formule $w = \sum r / (n + \lambda)$ ?
**Réponse.** La feuille choisit la correction $w$ qui minimise $\text{coût}(w) = \sum (r_i - w)^2 + \lambda w^2$. En annulant la dérivée : $-2\sum(r_i - w) + 2\lambda w = 0$ → $w = \sum r_i/(n + \lambda)$.
Feuille gauche de l’arbre 1 (résidus −30 et −20) : on calcule le coût de plusieurs corrections $w$ et on compare au minimum donné par la formule.

In [ ]:
r_feuille = [-30, -20]
cout = lambda w, lam: sum((r - w) ** 2 for r in r_feuille) + lam * w ** 2
for w_essai in [0, -10, -50 / 3, -20, -25]:
    print(f"w = {w_essai:>6.1f} : coût(λ=0) = {cout(w_essai, 0):7.1f}   coût(λ=1) = {cout(w_essai, 1):7.1f}")
for lam in [0, 1, 10]:
    w_opt = sum(r_feuille) / (len(r_feuille) + lam)
    print(f"λ = {lam:>2} : feuille optimale w = {w_opt:.2f} → nouvelle prédiction F1 = 40 + 0,5 × w = {40 + 0.5 * w_opt:.2f}")

w =    0.0 : coût(λ=0) =  1300.0   coût(λ=1) =  1300.0
w =  -10.0 : coût(λ=0) =   500.0   coût(λ=1) =   600.0
w =  -16.7 : coût(λ=0) =   188.9   coût(λ=1) =   466.7
w =  -20.0 : coût(λ=0) =   100.0   coût(λ=1) =   500.0
w =  -25.0 : coût(λ=0) =    50.0   coût(λ=1) =   675.0
λ =  0 : feuille optimale w = -25.00 → nouvelle prédiction F1 = 40 + 0,5 × w = 27.50
λ =  1 : feuille optimale w = -16.67 → nouvelle prédiction F1 = 40 + 0,5 × w = 31.67
λ = 10 : feuille optimale w = -4.17 → nouvelle prédiction F1 = 40 + 0,5 × w = 37.92


### Question 7 — MSE après chaque arbre pour λ = 0, 1 et 10
**Réponse.** Plus λ est grand, plus les corrections sont petites : il faut plus d’arbres pour la même erreur, mais le modèle colle moins au bruit.

In [ ]:
mses_lambda = {}
for lam in [0, 1, 10]:
    _, _, m_vrai = boosting_regression(T, Y, 3, 0.5, lam)
    mses_lambda[lam] = m_vrai
    print(f"λ = {lam:>2} : MSE après 0, 1, 2, 3 arbres = {[round(m, 1) for m in m_vrai]}")
print("Classification, feuille gauche de l'arbre 1 : λ = 0 →", -1 / 0.5, "; λ = 1 →", round(-1 / 1.5, 2))

λ =  0 : MSE après 0, 1, 2, 3 arbres = [650.0, 181.2, 64.1, 32.4]
λ =  1 : MSE après 0, 1, 2, 3 arbres = [650.0, 302.8, 148.5, 79.9]
λ = 10 : MSE après 0, 1, 2, 3 arbres = [650.0, 550.2, 466.3, 395.8]
Classification, feuille gauche de l'arbre 1 : λ = 0 → -2.0 ; λ = 1 → -0.67


---
# Partie B — Les mêmes exemples avec la bibliothèque xgboost

### Question 8 — `XGBRegressor` et `XGBClassifier`
**Réponse.** On règle la bibliothèque comme dans le cours :
- `n_estimators=3`, `learning_rate=0.5` (η), `max_depth=1`, `reg_lambda=λ` ;
- `base_score` : prédiction de départ $F_0$ (40 en régression ; 0,5 en classification, soit $F_0 = 0$) ;
- `min_child_weight=0` : par défaut, xgboost refuse une feuille dont la somme des $p(1-p)$ est inférieure à 1. Ici les feuilles sont très petites (4 exemples), on lève donc cette limite ;
- `tree_method="exact"` : tous les seuils possibles sont testés, comme dans notre code.

`get_booster().get_dump()` affiche les arbres appris (question et valeur des feuilles, déjà multipliées par η).

In [ ]:
import numpy as np, xgboost as xgb

Xt = np.array(T, dtype=float).reshape(-1, 1)
reg = xgb.XGBRegressor(n_estimators=3, learning_rate=0.5, max_depth=1, reg_lambda=0, base_score=40,
                       min_child_weight=0, tree_method="exact").fit(Xt, Y)
for k, d in enumerate(reg.get_booster().get_dump(), start=1):
    print(f"Arbre {k} :", d.replace("\n", " ").replace("\t", ""))
pred_reg = reg.predict(Xt)
print("Prédictions après 3 arbres :", pred_reg, "| MSE =", round(float(np.mean((np.array(Y) - pred_reg) ** 2)), 2))

mse_xgb_lambda = {}
for lam in [0, 1, 10]:
    m = xgb.XGBRegressor(n_estimators=3, learning_rate=0.5, max_depth=1, reg_lambda=lam, base_score=40,
                         min_child_weight=0, tree_method="exact").fit(Xt, Y)
    mse_xgb_lambda[lam] = float(np.mean((np.array(Y) - m.predict(Xt)) ** 2))
    print(f"λ = {lam:>2} : MSE après 3 arbres = {mse_xgb_lambda[lam]:.1f}")

Arbre 1 : 0:[f0<25] yes=1,no=2,missing=1 1:leaf=-12.5 2:leaf=12.5 
Arbre 2 : 0:[f0<25] yes=1,no=2,missing=1 1:leaf=-6.25 2:leaf=6.25 
Arbre 3 : 0:[f0<35] yes=1,no=2,missing=1 1:leaf=-1.875 2:leaf=5.625 
Prédictions après 3 arbres : [19.375 19.375 56.875 64.375] | MSE = 32.42
λ =  0 : MSE après 3 arbres = 32.4
λ =  1 : MSE après 3 arbres = 79.9
λ = 10 : MSE après 3 arbres = 395.8


In [ ]:
Xx = np.array(Xc, dtype=float).reshape(-1, 1)
clf = xgb.XGBClassifier(n_estimators=3, learning_rate=0.5, max_depth=1, reg_lambda=0, base_score=0.5,
                        min_child_weight=0, tree_method="exact").fit(Xx, Yc)
for k, d in enumerate(clf.get_booster().get_dump(), start=1):
    print(f"Arbre {k} :", d.replace("\n", " ").replace("\t", ""))
p_xgb = clf.predict_proba(Xx)[:, 1]
print("p après 3 arbres :", p_xgb.round(3), "| p(3,5 h) =", round(float(clf.predict_proba([[3.5]])[0, 1]), 3))

Arbre 1 : 0:[f0<2.5] yes=1,no=2,missing=1 1:leaf=-1 2:leaf=1 
Arbre 2 : 0:[f0<2.5] yes=1,no=2,missing=1 1:leaf=-0.683939695 2:leaf=0.683939695 
Arbre 3 : 0:[f0<2.5] yes=1,no=2,missing=1 1:leaf=-0.592820585 2:leaf=0.592820585 
p après 3 arbres : [0.093 0.093 0.907 0.907] | p(3,5 h) = 0.907


---
# Partie C — Comparaison et validation
### Question 9 — Les résultats concordent-ils ?

In [ ]:
import pandas as pd
lignes = [(f"Ex.1 prédiction à {t} °C après 3 arbres", round(predire_boost(t, F0, arbres_vrai, 0.5), 4), round(float(v), 4))
          for t, v in zip(T, pred_reg)]
lignes += [(f"Ex.2 p(réussite) pour {x} h après 3 arbres", round(hist[3][1][i], 4), round(float(p_xgb[i]), 4)) for i, x in enumerate(Xc)]
lignes += [(f"Ex.3 MSE après 3 arbres, λ = {lam}", round(mses_lambda[lam][-1], 3), round(mse_xgb_lambda[lam], 3)) for lam in [0, 1, 10]]
display(pd.DataFrame(lignes, columns=["Grandeur", "Python pur", "bibliothèque xgboost"]))
assert np.allclose([predire_boost(t, F0, arbres_vrai, 0.5) for t in T], pred_reg, atol=1e-4)
assert np.allclose(hist[3][1], p_xgb, atol=1e-5)
assert all(abs(mses_lambda[l][-1] - mse_xgb_lambda[l]) < 1e-3 for l in [0, 1, 10])
print("✅ Notre XGBoost en Python pur donne exactement les mêmes résultats que la bibliothèque xgboost")

,Grandeur,Python pur,bibliothèque xgboost
0,Ex.1 prédiction à 10 °C après 3 arbres,19.3750,19.3750
1,Ex.1 prédiction à 20 °C après 3 arbres,19.3750,19.3750
2,Ex.1 prédiction à 30 °C après 3 arbres,56.8750,56.8750
3,Ex.1 prédiction à 40 °C après 3 arbres,64.3750,64.3750
4,Ex.2 p(réussite) pour 1 h après 3 arbres,0.0931,0.0931
5,Ex.2 p(réussite) pour 2 h après 3 arbres,0.0931,0.0931
6,Ex.2 p(réussite) pour 3 h après 3 arbres,0.9069,0.9069
7,Ex.2 p(réussite) pour 4 h après 3 arbres,0.9069,0.9069
8,"Ex.3 MSE après 3 arbres, λ = 0",32.4220,32.4220
9,"Ex.3 MSE après 3 arbres, λ = 1",79.8700,79.8700


✅ Notre XGBoost en Python pur donne exactement les mêmes résultats que la bibliothèque xgboost


## Conclusion



- Le boosting construit les arbres **l’un après l’autre** : chacun apprend les erreurs laissées par les précédents.
- La valeur d’une feuille est $\sum r/(n + \lambda)$ en régression, $\sum (y-p)/(\sum p(1-p) + \lambda)$ en classification.
- **À chaque arbre**, la question est recalculée : elle peut changer d’un arbre à l’autre.
- η et λ rendent les corrections plus prudentes ; on les règle par validation croisée.